# AML Alert Quality, QA & Rule Performance Analytics


In [ ]:
import pandas as pd
alerts=pd.read_csv('../data/processed/alerts_enriched.csv')
qa=pd.read_csv('../data/processed/qa_reviews.csv')
rules=pd.read_csv('../data/processed/rule_performance.csv')
rules.head()


## EDA & Feature Engineering Workflow

This section adds a clear, practical EDA and feature engineering workflow before the existing AML alert QA and rule-performance analysis. The goal is to demonstrate data-quality checks, exploratory analysis, and simple AML-focused feature creation without making the project unnecessarily advanced.


In [ ]:
# 1. Dataset review
print('Alerts shape:', alerts.shape)
print('QA reviews shape:', qa.shape)
print('Rules shape:', rules.shape)

print('\nAlerts columns:')
print(alerts.columns.tolist())
print('\nRules columns:')
print(rules.columns.tolist())

alerts.head()


In [ ]:
# 2. Missing values and duplicate validation
for name, df in {'alerts': alerts, 'qa': qa, 'rules': rules}.items():
    print(f'\n{name.upper()}')
    print('Duplicate rows:', df.duplicated().sum())
    missing = df.isna().sum()
    print('Missing values:')
    print(missing[missing > 0] if (missing > 0).any() else 'No missing values')


In [ ]:
# 3. Datatype and column-name review
for name, df in {'alerts': alerts, 'qa': qa, 'rules': rules}.items():
    # Standardize column names without changing the business meaning
    df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False))
    print(f'\n{name.upper()} datatypes')
    print(df.dtypes)


In [ ]:
# 4. Basic data-quality and range checks
# Check common numeric fields only when they exist in the dataset.
for name, df in {'alerts': alerts, 'qa': qa, 'rules': rules}.items():
    numeric_cols = df.select_dtypes(include='number').columns
    print(f'\n{name.upper()} numeric summary')
    if len(numeric_cols):
        display(df[numeric_cols].describe().T)
    else:
        print('No numeric columns found.')


In [ ]:
# 5. Simple outlier review using the IQR method
# This flags unusual values for investigation; it does not automatically remove them.
def iqr_outlier_count(series):
    series = series.dropna()
    if series.empty:
        return 0
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in rules.select_dtypes(include='number').columns:
    print(f'{col}: {iqr_outlier_count(rules[col])} potential outliers')


In [ ]:
# 6. KPI validation
# Confirm that percentage/rate fields are within reasonable boundaries.
rate_cols = [c for c in rules.columns if 'rate' in c.lower()]
for col in rate_cols:
    values = rules[col].dropna()
    if not values.empty:
        print(f'{col}: min={values.min():.2f}, max={values.max():.2f}')


In [ ]:
# 7. Feature engineering
# Create simple, interpretable features useful for QA and rule-performance review.
rules_fe = rules.copy()

if {'alert_volume', 'false_positive_rate'}.issubset(rules_fe.columns):
    rules_fe['estimated_false_positive_alerts'] = (
        rules_fe['alert_volume'] * rules_fe['false_positive_rate']
    ).round(0)

if {'qa_pass_rate'}.issubset(rules_fe.columns):
    rules_fe['qa_failure_rate'] = 1 - rules_fe['qa_pass_rate']

if {'false_positive_rate', 'sar_conversion_rate'}.issubset(rules_fe.columns):
    rules_fe['rule_review_flag'] = (
        (rules_fe['false_positive_rate'] >= rules_fe['false_positive_rate'].median()) &
        (rules_fe['sar_conversion_rate'] <= rules_fe['sar_conversion_rate'].median())
    )

new_features = [c for c in ['estimated_false_positive_alerts','qa_failure_rate','rule_review_flag'] if c in rules_fe.columns]
print('Engineered features:', new_features)
rules_fe[['rule_name'] + new_features].head() if new_features else rules_fe.head()


In [ ]:
# 8. Business-rule validation
# Review rules that may deserve analyst attention based on the engineered flag.
if 'rule_review_flag' in rules_fe.columns:
    display(rules_fe[rules_fe['rule_review_flag']].sort_values('false_positive_rate', ascending=False))
else:
    print('Required fields for rule_review_flag are not available.')


In [ ]:
# 9. Final EDA summary
print('Rules analyzed:', len(rules_fe))
if 'alert_volume' in rules_fe.columns:
    print('Total alert volume:', rules_fe['alert_volume'].sum())
if 'false_positive_rate' in rules_fe.columns:
    print('Average false-positive rate:', round(rules_fe['false_positive_rate'].mean(), 3))
if 'qa_pass_rate' in rules_fe.columns:
    print('Average QA pass rate:', round(rules_fe['qa_pass_rate'].mean(), 3))
if 'rule_review_flag' in rules_fe.columns:
    print('Rules flagged for review:', int(rules_fe['rule_review_flag'].sum()))


### EDA / Feature Engineering Takeaway

The workflow validates data quality, reviews distributions and potential outliers, checks KPI ranges, and creates interpretable QA/rule-performance features. Potential outliers and weak-performing rules are treated as investigation candidates rather than automatically removed, which preserves relevant AML behavior for further review.


## False positive and escalation performance

In [ ]:
rules[['rule_name','alert_volume','false_positive_rate','escalation_rate','sar_conversion_rate']].sort_values('false_positive_rate',ascending=False)


## QA performance

In [ ]:
rules[['rule_name','qa_reviews','qa_pass_rate']].sort_values('qa_pass_rate')


## Analyst-level rule review candidates

In [ ]:
rules[rules['tuning_recommendation']!='Monitor'].sort_values('tuning_priority_score',ascending=False)
